In [6]:
import torch

import triton
import triton.language as tl

DEVICE = triton.runtime.driver.active.get_active_torch_device()


In [ ]:
"""Define a function that runs on the GPU using triton"""
@triton.jit
def test(
    x_ptr, # pointer* to the first input vector
    y_ptr, # Pointer* to the second input vector
    output_ptr, # Pointer to the output vector
    n_elements, # size of the vector
    BLOCK_SIZE: tl.constexpr # Number of elemts each program should process
    #  NOTE: `constexpr` so it can be used as a shape value.
):
    # Identify which program to process data
    pid = tl.program_id(axis=0) # We use a 1D launch grid so axis is 0.

    # offset is a list of pointers
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    # Create a mask to guard memory operations against out-of-bounds accesses.
    mask = offsets < n_elements
    # Load x and y from DRAM, masking out any extra elements in case the input is not a
    # multiple of the block size.
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    output = x + y
    # Write x + y back to DRAM.
    tl.store(output_ptr + offsets, output, mask=mask)


In [ ]:
@triton.jit
def add_kernel(x_ptr,  # *Pointer* to first input vector.
               y_ptr,  # *Pointer* to second input vector.
               output_ptr,  # *Pointer* to output vector.
               n_elements,  # Size of the vector.
               BLOCK_SIZE: tl.constexpr,  # Number of elements each program should process.
               # NOTE: `constexpr` so it can be used as a shape value.
               ):
    # There are multiple 'programs' processing different data. We identify which program
    # we are here:
    pid = tl.program_id(axis=0)  # We use a 1D launch grid so axis is 0.
    # This program will process inputs that are offset from the initial data.
    # For instance, if you had a vector of length 256 and block_size of 64, the programs
    # would each access the elements [0:64, 64:128, 128:192, 192:256].
    # Note that offsets is a list of pointers: 
    """
    Program pid = 0 => block_start = 0 * 4 = 0
    Program pid = 1 => block_start = 1 * 4 = 4
    """
    block_start = pid * BLOCK_SIZE
    """
    tl.arange(0, BLOCK_SIZE)
    block_start = 0
    offsets = 0 + [0,1,2,3]
    block_start = 4
    offsets = 4 + [0,1,2,3]
    """
    # offsets là một Triton tensor (tl.tensor)
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    # Create a mask to guard memory operations against out-of-bounds accesses.
    """
    Triton tensor(tl.tensor) chứa giá trị boolean."""
    mask = offsets < n_elements
    # Load x and y from DRAM, masking out any extra elements in case the input is not a
    # multiple of the block size.
    """
    x_ptr  # Pointer to beginning of [1, 2, 3, 4, 5, 6, 7, 8]
    offsets = [4, 5, 6, 7]
    x_ptr + offsets -> produces pointers to 
    the elements at indices 4, 5, 6 and 7.
    """
    """
    Triton biết block_start và offsets nhờ pid.
    Mỗi Triton program được gán một pid khác nhau.
    Vì vậy, dù tất cả program chạy cùng một đoạn code,
    chúng sẽ tính ra các offsets khác nhau.
    """
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    output = x + y
    # Write x + y back to DRAM.
    tl.store(output_ptr + offsets, output, mask=mask)

In [ ]:
def add(x: torch.Tensor, y: torch.Tensor):
    output = torch.empty_like(x)

    n_elements = output.numel()

    """Defines how many GPU blocks (kernel instances)
    Triton should launch to process all elements of tensor"""
    """
    lambda: define function without a name
    def grid(meta):
        return (triton.cdiv(n_elements, meta['BLOCK_SIZE']), )
    
    """
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']), )

    """Specify how many instances of the GPU kernel to launch"""
    """grid is a tuple. e.g: (3,)"""
    add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=1024) # type: ignore

    return output

In [9]:
torch.manual_seed(0)
size = 98432
x = torch.randn(size, device=DEVICE)
y = torch.randn(size, device=DEVICE)
output_torch = x + y
output_triton = add(x,y)
print(output_torch)
print(output_triton)

tensor([-0.7438, -0.9776, -1.7200,  ..., -2.0104, -1.5577,  0.7311],
       device='cuda:0')
tensor([-0.7438, -0.9776, -1.7200,  ..., -2.0104, -1.5577,  0.7311],
       device='cuda:0')
